# N1 · SciBERT Named Entity Recognition (SciERC)
NLP track, model 1. Fine-tunes `allenai/scibert_scivocab_uncased` to tag **Task, Method, Metric, Material,
OtherScientificTerm, Generic** in scientific text. Baseline: general-domain `bert-base-uncased`.
Every run saves `metrics.json`, `history.csv`, `predictions.jsonl` to Drive and resumes automatically after a disconnect.

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Add `HF_TOKEN` in the Secrets panel (key icon) with notebook access on.
Edit the two settings below once.

In [ ]:
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
HF_USERNAME = "colab"                                          # <- change
DRIVE_ROOT = "/content/drive/MyDrive/ResearchLens"                        # checkpoints + results live here

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.exists("/content/researchlens"):
    !git clone -q {REPO_URL} /content/researchlens
else:
    !git -C /content/researchlens pull -q
%cd /content/researchlens
!pip install -q -r requirements-colab.txt

os.environ["HF_USERNAME"] = HF_USERNAME
os.environ["RL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TQDM_DISABLE"] = "1"   # plain log lines instead of progress bars
# Colab Secrets are only readable here in the notebook, so pass the token on to the training scripts.
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret not found / notebook access off -> models will not be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess, sys
def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished (metrics.json on Drive): safe to re-run after a disconnect.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    done = f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"
    if os.path.exists(done):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:            # stream the training log live into the notebook
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. Train SciBERT (3 seeds; seed 42 is pushed to the Hugging Face Hub)

In [ ]:
SEEDS = [42, 43, 44]
for s in SEEDS:
    run("researchlens.nlp.train_ner", "configs/n1_ner.yaml", "scibert", s, "--push_to_hub" if s == 42 else "")

## 2. Baselines
- **BERT-base** with the same regularised settings: shows the value of scientific pre-training.
- **SciBERT without regularisation** (the original settings: lr 3e-5, 10 epochs, weight decay 0.01, dropout 0.1,
  embeddings trainable, patience 3): shows how much the anti-over-fitting settings reduce the train/test gap.

In [ ]:
for s in SEEDS:
    run("researchlens.nlp.train_ner", "configs/n1_ner.yaml", "bert_base", s, "--model_name bert-base-uncased")
NOREG = ("--lr 3e-5 --epochs 10 --weight_decay 0.01 --hidden_dropout 0.1 --attention_dropout 0.1 "
         "--no-freeze_embeddings --early_stopping_patience 3")
for s in SEEDS:
    run("researchlens.nlp.train_ner", "configs/n1_ner.yaml", "scibert_noreg", s, NOREG)

## 3. Report: all graphs + tables

In [ ]:
from researchlens.evaluation.reports import ner_report
REPORT_DIR = f"{DRIVE_ROOT}/results/n1_ner/report"
display(ner_report(f"{DRIVE_ROOT}/results"))

from pathlib import Path
from IPython.display import Image, display
for f in sorted(Path(REPORT_DIR).glob("*.png")):
    print(f.name); display(Image(str(f), width=650))